In [ ]:
import tensorflow as tf
import numpy as np
from utils import *
import keras as kr

In [ ]:
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y,R = load_ratings_small() # Y => Actual Rating ( if there is no rating it is considered to be 0 )
                           # R => Is rating exists ?

In [ ]:
ratings = Y[R == 1] # It stores those ratings in ratings array where that actually exist (R==1 means if rating exists)
mean = np.mean(ratings)

<h1> Cost Function (Standard)</h1>

In [ ]:
def calc_cost(X, W, B, Y, R, lambda_):
    nm,nu  = Y.shape
    J = 0
    for j in range(nu):
        w = W[j,:] # select all colmns of row 1
        b = B[0,j] # select i column of first row because it has only one rows
        for i in range(nm):
            y = Y[i,j] # For movie (i) get the user (j) rating
            r = R[i,j] # For movie (i) get the user (j) rating if exist
            x = X[i,:] # For movie (i) get all the features like how much it is (romantic,action,thriller etc)
            J += r*(np.square(np.dot(w,x)+b-y)) # We are multiplying by r to full fill the "r(i,j)==1" condition of cost function formula means consider only the error for those where user j has rated the movie i
    J+=(lambda_)*(np.sum(np.square(W))+np.sum(np.square(X)))
    J= J/2
    return J


<h1> Cost Function (vectorized)</h1>

In [ ]:
def vec_cost (X, W, B, Y, R, lambda_ ):
    # We are multiplying by r to full fill the "r(i,j)==1" condition of cost function formula means consider only the error for those where user j has rated the movie i
    J = ( tf.linalg.matmul(X,tf.transpose(W))+B - Y )*R # This returns an Array also we are taking transpose so that dimension should match for multiplication(AxB = BxA)
    regularization = (lambda_/2)*(tf.reduce_sum(W**2) + tf.reduce_sum(X**2))
    J = 0.5*(tf.reduce_sum(J**2)) + regularization
    return J

<h1>Rating the Movies</h1>

In [ ]:
movieList, movieList_df = load_Movie_List_pd()
my_ratings = np.zeros(num_movies)# We are generating and np array of zeros of length equal to num of movies so that we have to rate only those movies which i like and the other movies rating will be zero automatically

# Now we are rating the Movies we like by replacing the zero with our rating

my_ratings[929]  = 2   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 5   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)

my_rated = [i for i in range(len(my_ratings)) if my_ratings[i]>0] # getting the index of movies which we rated and putting it inside array

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');

<h1>Adding new column of my ratings in data set</h1>

In [ ]:
Y,R = load_ratings_small()
Y = np.c_[my_ratings,Y] # Add new column my_rating in the Y(all user rating) at start
R = np.c_[(my_ratings != 0 ).astype(int),R] # Adding column of (ones and zeros) in R that tells that did my rating exist for movie
# normalizing the data
Ynorm, Ymean = normalizeRatings(Y, R)

<h1>Training model</h1>

In [ ]:
# maing tensorflow variables
num_movies,num_users = Y.shape
num_features = 100
tf.random.set_seed(1234)
'''
Below the tf.random.normal(num_users,num_features) produce random " guassian (normal) distribution values "
which are very small like 0.5,0.6 etc. We do this so that our starting parameters like weight and bias are
not too big like 500,300 etc which can effect our model learning
'''
W = tf.Variable(tf.random.normal((num_users,num_features),dtype=tf.float64),name="W")
B = tf.Variable(tf.random.normal((1,num_users),dtype=tf.float64),name="B")
X = tf.Variable(tf.random.normal((num_movies,num_features),dtype=tf.float64),name="X")
optimizer = kr.optimizers.Adam(learning_rate=1e-1)

In [ ]:
def train_model(X, W, B, Ynorm, R, lambda_):
    iterations = 1000
    for iter in range(iterations):
        '''
        The gradient tape just calculate the cost and remembers the steps(how it was calculated) and store
        it in tape variable so that the below tape.gradient function just implement all the steps backward to
        calculate gradient same as andrew told in derivatives lecture
        '''
        with tf.GradientTape() as tape : # storing the steps info into tape so that we can calculate gradient later
            cost = vec_cost(X,W,B,Ynorm,R,lambda_)
        grads = tape.gradient(cost,[X,W,B]) # calculating the gradients of [X,W,B] with respect to cost and store it in list(grads)
        optimizer.apply_gradients(zip(grads,[X,W,B])) # as the grads contain the list of gradients we pair the gradient with their name [X,W,B]
        '''
        conceptually the zip is doing this
        grads contain gradients like that :
        grads[0] → gradient of X
        grads[1] → gradient of W
        grads[2] → gradient of b
        zip function paris them as :
        gradient_X → X
        gradient_W → W
        gradient_b → b
        '''
        if iter % 20 == 0:
            print(f"Training loss at iteration {iter}: {cost:0.1f}")
    return X,W,B



In [ ]:
lambda_ = 1
numbers = train_model(X,W,B,Ynorm,R,lambda_)

In [ ]:

X = numbers[0]
W = numbers[1]
B = numbers[2]

p = np.matmul(X.numpy(),np.transpose(W.numpy())) + B.numpy()
"""
we are taking transpose of weights because the weights are aligned like tht :
we have weights for each user and for each feature,means data set is aligned like columns show the number of features and rows shows number of users.

       feature 1 | feature 2 | feature 3 ............
user 1
user 2
user 3
.
.
.
.
.
"""
# Now adding the mean back into prediction
p_after_adding_mean = p + Ymean
my_predictions = p_after_adding_mean[:,0] # This means gave me predictions ratings on movies for user 0 which is mine
arr_preds = tf.argsort(my_predictions,direction="DESCENDING") # This returns the index of movie ratings (after sorting it) not the movie rating itself. Tensor-flow sort() function returns items and tensor-flow argsort() returns indexes

for i in range(100) :
  j = arr_preds[i] # here arr_preds[i] is returning the index not movie rating
  if j not in my_rated :
      print(f"Predicted Rating : {my_predictions[j]:0.2f} | Movie : {movieList[j]} ")


In [ ]:
print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')